In [1]:
import re
import time
from urllib.parse import urljoin, urlsplit, urlunsplit

import pandas as pd
import requests
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

from datetime import datetime
from zoneinfo import ZoneInfo

import html

In [2]:
BASE_URL = "https://www.inven.co.kr"
BOARD_URL = "https://www.inven.co.kr/board/maple/2300"
START_PAGE = 1
END_PAGE = 85
REQUEST_DELAY = 1.0

In [3]:

# 세션 설정
session = requests.Session()

session.headers.update(
    {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/139.0 Safari/537.36"
        ),
        "Accept-Language": "ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7",
    }
)

retry = Retry(
    total=3,
    connect=3,
    read=3,
    backoff_factor=1,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods={"GET", "POST"},
)

adapter = HTTPAdapter(max_retries=retry)

session.mount("https://", adapter)
session.mount("http://", adapter)

# 게시글 저장용 리스트와 중복 방지용 세트
all_posts = []
visited_urls = set()

In [4]:
COMMENT_URL = "https://www.inven.co.kr/common/board/comment.json.php"
BOARD_CODE = "2300"

def fetch_comments(session, article_url):
    article_code = urlsplit(article_url).path.rstrip("/").split("/")[-1]

    offset = 0
    page_size = 100

    while True:
        payload = {
            "comeidx": BOARD_CODE,
            "articlecode": article_code,
            "sortorder": "date",
            "act": "list",
            "out": "json",
            "replynick": "",
            "replyidx": 0,
            "uploadurl": "",
            "imageposition": "",
            "videoloading": "lazy",
        }

        if offset > 0:
            payload["titles"] = offset

        response = session.post(
            COMMENT_URL,
            params={
                "dummy": int(time.time() * 1000)
            },
            data=payload,
            headers={
                "Referer": article_url,
                "X-Requested-With": "XMLHttpRequest",
            },
            timeout=15,
        )

        response.raise_for_status()
        data = response.json()
        comments = [] 

        # 댓글 추출
        for comment_group in data.get("commentlist", []):
            for comment in comment_group.get("list", []):

                attr = comment.get("__attr__", {})

                # HTML entity 복원
                content = comment.get("o_comment", "")

                content = html.unescape(
                    html.unescape(content)
                ).replace("\xa0", " ")

                comments.append({
                    "comment_id": attr.get("cmtidx"),
                    "parent_id": attr.get("cmtpidx"),
                    "author": comment.get("o_name"),
                    "created_at": comment.get("o_date"),
                    "content": comment.get("o_comment"),
                    "recommend": comment.get("o_recommend"),
                    "not_recommend": comment.get("o_notrecommend"),
                })

        total_count = int(data.get("cmtcount", 0))

        if len(comments) >= total_count:
            break

        offset += page_size

    return comments

In [5]:
# 페이지별 수집
for page in range(START_PAGE, END_PAGE + 1):
    print("\n" + "=" * 60)
    print(f"{page}페이지 수집 시작")
    print("=" * 60)

    try:
        # 게시판 목록 페이지 요청
        response = session.get(
            BOARD_URL,
            params={"p": page},
            timeout=15
        )
        response.raise_for_status()

        if response.encoding is None or response.encoding.lower() == "iso-8859-1":
            response.encoding = response.apparent_encoding

        soup = BeautifulSoup(response.text, "html.parser")

        # 게시글 링크 추출
        articles = soup.select(".text-wrap .subject-link")
        print(f"[페이지 {page}] 게시글 링크 후보: {len(articles)}개")

        post_urls = []

        for article in articles:
            href = article.get("href")

            if not href:
                continue

            url = urljoin(BASE_URL, href)
            parsed = urlsplit(url)

            if not re.fullmatch(r"/board/maple/2300/\d+", parsed.path):
                continue

            clean_url = urlunsplit(
                (
                    parsed.scheme,
                    parsed.netloc,
                    parsed.path,
                    "",
                    ""
                )
            )

            post_urls.append(clean_url)

        post_urls = list(dict.fromkeys(post_urls))
        print(f"수집할 게시글: {len(post_urls)}개")

    except requests.RequestException as exc:
        print(f"[페이지 요청 실패] page={page}")
        print(exc)
        continue


    # 게시글 상세 내용 수집
    for index, url in enumerate(post_urls, start=1):
        if url in visited_urls:
            continue

        try:
            response = session.get(url, timeout=15)
            response.raise_for_status()

            if response.encoding is None or response.encoding.lower() == "iso-8859-1":
                response.encoding = response.apparent_encoding

            soup = BeautifulSoup(response.text, "html.parser")

            title_element = soup.select_one(".articleTitle")
            author_element = soup.select_one(".nickname")
            date_element = soup.select_one(".articleDate")
            content_element = soup.select_one(".contentBody")
            category_element = soup.select_one(".articleCategory")

            if title_element is None:
                raise ValueError(f"title을 찾지 못했습니다: {url}")

            if content_element is None:
                raise ValueError(f"content를 찾지 못했습니다: {url}")

            # ==========================================
            # 댓글 데이터 수집
            # ==========================================

            comments = fetch_comments(
                session=session,
                article_url=url,
            )


            # ==========================================
            # 기존 게시글 정보 처리
            # ==========================================

            info_text = ""

            if date_element:
                node = date_element

                while node and getattr(node, "name", None) != "body":
                    text = node.get_text(" ", strip=True)

                    if "조회:" in text and "추천:" in text:
                        info_text = text
                        break

                    node = node.parent

            if category_element:
                category = (
                    category_element
                    .get_text(" ", strip=True)
                    .strip()
                    .strip("[]")
                    .strip()
                )
            else:
                category_match = re.search(r"\[([^\]]+)\]", info_text)

                category = (
                    category_match.group(1).strip()
                    if category_match
                    else None
                )

            # 4. views
            views_match = re.search(
                r"조회:\s*([\d,]+)",
                info_text
            )

            views = (
                int(views_match.group(1).replace(",", ""))
                if views_match
                else None
            )


            # 5. likes
            likes_match = re.search(
                r"추천:\s*([\d,]+)",
                info_text
            )

            likes = (
                int(likes_match.group(1).replace(",", ""))
                if likes_match
                else None
            )

            # 6. 결과 저장
            post = {
                "url": url,
                "category": category,
                "title": title_element.get_text(" ", strip=True),

                "author": (
                    author_element.get_text(" ", strip=True)
                    if author_element
                    else None
                ),

                "created_at": (
                    date_element.get_text(" ", strip=True)
                    if date_element
                    else None
                ),

                "views": views,
                "likes": likes,

                "content": content_element.get_text(
                    "\n",
                    strip=True
                ),

                "comment_count": len(comments),
                "comments": comments,
                

                "crawled_at": datetime.now(
                    ZoneInfo("Asia/Seoul")
                ).isoformat(timespec="seconds"),
            }

            all_posts.append(post)
            visited_urls.add(url)

            print(
                f"[{page}페이지 {index}/{len(post_urls)}] "
                f"{post['title']}"
            )

        except requests.RequestException as exc:
            print(f"[HTTP 오류] {url}")
            print(exc)

        except Exception as exc:
            print(f"[파싱 오류] {url}")
            print(exc)

        time.sleep(REQUEST_DELAY)

    time.sleep(REQUEST_DELAY)


1페이지 수집 시작
[페이지 1] 게시글 링크 후보: 51개
수집할 게시글: 51개
[1페이지 1/51] 질문 게시글 삭제 및 수정에 대한 이용 안내
[1페이지 2/51] 모멘텀패스 살까요 플러스 나오는 거 살까요?
[1페이지 3/51] 인벤 계정 레벨 어떻게 올리는거에요?
[1페이지 4/51] 자석펫 확률업 공지 하고 하나요??
[1페이지 5/51] 울티마 상점떔에 3배 썩어나는데 4배로 교환할수있나요??
[1페이지 6/51] 뉴비 템세팅 질문.....
[1페이지 7/51] 메이플 결정석 수수료있나요...?
[1페이지 8/51] 제네시스 패스
[1페이지 9/51] 1석->2석 순수자석범위는 같나요?
[1페이지 10/51] 제네플
[1페이지 11/51] 안녕하세요 하드메이린 세팅 문의드립니다
[1페이지 12/51] 17성 에테 유에
[1페이지 13/51] 챌섭 지금 시점에서 뉴비가 검마 2~3인트라이하려면 어떤구인방법이 있을까요
[1페이지 14/51] 샤타때 만들어버렸는데 이후 템셋 문의… 9-10만목표
[1페이지 15/51] 매쌤들 질문있슴당
[1페이지 16/51] 흙테르넬 질문
[1페이지 17/51] 똥손도 최소컷 쉬운 직업
[1페이지 18/51] 뎀스 가격 질문입니다.
[1페이지 19/51] 이거 왜 큐브 안돌려지죠?
[1페이지 20/51] 메m 250 7일안에 가능한가요?
[1페이지 21/51] 챌섭 질문 있습니다.
[1페이지 22/51] 챌섭 4주안에 챌린저 힘들가요?
[1페이지 23/51] 엔버 링크 서버렉 먹음?
[1페이지 24/51] 파일손상재설치 관련
[1페이지 25/51] 똥블렘->미트라 어느 수준으로 맞출까요?
[1페이지 26/51] 150만원으로 템세팅vs 5석펫
[1페이지 27/51] 아버 미션 노말보스 하드로 깨도 되나요?
[1페이지 28/51] 렌 제네무기 추옵 질문입니다!!!!!
[1페이지 29/51] 하드메이린 104프로 빌드 충고 부탁드립니다.
[1페이지 30/51] 노적자 노칼 목표 스펙업 방향부탁드립니다
[1페이지 31/5

In [7]:
columns = [
    "url",
    "category",
    "title",
    "author",
    "created_at",
    "views",
    "likes",
    "content",
    "comment_count",
    "comments",
    "crawled_at",
]

df = pd.DataFrame(
    all_posts,
    columns=columns
)

output_path = "../../../data/raw/maple_inven_questions_원본.csv"

df.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print(f"\n총 수집 게시글: {len(df)}개")
print(f"CSV 파일 저장 완료: {output_path}")

df.head()


총 수집 게시글: 4244개
CSV 파일 저장 완료: ../../../data/raw/maple_inven_questions_원본.csv


,url,category,title,author,created_at,views,likes,content,comment_count,comments,crawled_at
0,https://www.inven.co.kr/board/maple/2300/351627,기타,질문 게시글 삭제 및 수정에 대한 이용 안내,인벤운영팀,2026-01-22 17:30,5911,0,"안녕하세요,\n인벤\n입니다.\n커뮤니티 이용 과정에서 질문 게시글을 통해 도움을 ...",2,"[{'comment_id': 464645, 'parent_id': 464645, '...",2026-08-19T16:35:01+09:00
1,https://www.inven.co.kr/board/maple/2300/361956,기타,모멘텀패스 살까요 플러스 나오는 거 살까요?,지서어끄,2026-08-19 16:27,36,0,챌섭에서 렌 키우고 있습니다.\n지금 렙 284인데 모멘텀패스 사고 다음 주 에픽던...,1,"[{'comment_id': 497253, 'parent_id': 497253, '...",2026-08-19T16:35:03+09:00
2,https://www.inven.co.kr/board/maple/2300/361955,기타,인벤 계정 레벨 어떻게 올리는거에요?,수soo89,2026-08-19 16:19,28,0,오메나... 10렙을 목표로하고 있는데 이거 쉽지 않은거 맞지요?,0,[],2026-08-19T16:35:04+09:00
3,https://www.inven.co.kr/board/maple/2300/361954,기타,자석펫 확률업 공지 하고 하나요??,풀업30,2026-08-19 15:54,143,0,자석펫 사야 하는데\n혹시 내일 패치하면 들어올까 해서\n공지 하고 하나요??\n공...,0,[],2026-08-19T16:35:06+09:00
4,https://www.inven.co.kr/board/maple/2300/361953,아이템,울티마 상점떔에 3배 썩어나는데 4배로 교환할수있나요??,NaN,2026-08-19 15:48,157,0,받기전 3배 이벤창에서밖에 교환못함요...?,2,"[{'comment_id': 497248, 'parent_id': 497248, '...",2026-08-19T16:35:07+09:00
